In [18]:
import pandas as pd
import numpy as np

## **Pax agreements**

From all the set of conflicts that sign agreements, how many do we keep as treated in the panel.

1. Considering that an agreement is considered as 1 if conflict_active condition is met.

Muchos de los conflicts que firman acuerdos en PAX no son incluidos en el panel, porque firman acuerdos en momentos donde no se registra conflicto en UCDP y muchos conflict_id de PAX no están en UCDP.

Esos dos motivos explican la diferencia entre 115 conflics id en PAX y 71 conflict id en el panel con agreement = 1.

In [19]:
df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv', low_memory=False)
df_pax.columns = df_pax.columns.str.lower()

df_pax = df_pax.rename(columns={"ucdpcon": "conflict_id", "loc1iso": "isocode"})

df_pax["dat"] = pd.to_datetime(df_pax["dat"])
df_pax["year_mo"] = df_pax["dat"].dt.to_period("M").dt.to_timestamp()

# Replace SSD -> SDN before July 2011
mask = (df_pax["isocode"] == "SSD") & (
    (df_pax["dat"].dt.year < 2011) |
    ((df_pax["dat"].dt.year == 2011) & (df_pax["dat"].dt.month <= 6))
)
df_pax.loc[mask, "isocode"] = "SDN"

df_pax = df_pax.dropna(subset=['conflict_id', 'year_mo'])
df_pax["conflict_id"] = df_pax["conflict_id"].astype(int)

#Treatment indicators
df_pax["agreement"] = 1


In [20]:
df_pax['conflict_id'].nunique()

115

In [21]:
df_pax.groupby('conflict_id')['agreement'].sum()

conflict_id
209       28
218        8
221        7
222        1
223        1
          ..
13243      1
13306      9
13324      1
14275      2
169170     6
Name: agreement, Length: 115, dtype: int64

In [22]:
#Treatment indicators
df_pax["agreement"] = 1
df_pax["subs_agreement"] = (df_pax["stage"].str.lower() == "subpar").astype(int)
df_pax["comp_agreement"] = (df_pax["stage"].str.lower() == "subcomp").astype(int)
df_pax["cea_agreement"] = (df_pax["stage"].str.lower() == "cea").astype(int)
df_pax["cea_ceamix_agreement"] = (df_pax["stagesub"].str.lower() == "ceamix").astype(int)
df_pax["cea_ceas_agreement"] = (df_pax["stagesub"].str.lower() == "ceas").astype(int)
df_pax["cea_rel_agreement"] = (df_pax["stagesub"].str.lower() == "rel").astype(int)

#define variables to test clusterd Heterogenous TE
features_cluster_columns = ['hriimon', 'ime', 'ddrprog', 'imun', 'tjrep', 'ppsaut', 'tjmis', 
                            'protgrp', 'ppsvet', 'hriibod', 'tpsloc', 'terps', 'pol', 'epsres', 
                            'impk', 'epsoth', 'polnewtemp', 'protciv', 'hrtrinc', 'ssrddr', 
                            'ceprov', 'ppsint', 'ddrdemil', 'polpar', 'polps', 'hrbor', 'hrdem',
                            'medlog', 'tjvet', 'medsubs', 'epsfis', 'tpssub', 'tpsoth', 'eleccomm', 
                            'eps', 'ssrpol', 'medgov', 'imref', 'tpsaut', 'med', 'hrii', 'cegen', 'ssrgua',
                            'ssrarm', 'tjcou', 'ppsoro', 'tjamban', 'tjvic', 'tjmech', 'polnewind', 'mpsme', 
                            'mpsjt', 'imoth', 'ppsothpr', 'ppsex', 'hrmob', 'ele', 'hrni', 'civso', 'pubad', 
                            'juscr', 'mps', 'prot', 'mpspro']

df_agreements = (
    df_pax
    .groupby(["conflict_id", "year_mo"], as_index=False)
    .agg(
        agreement=("agreement", "max"),  
        comp_agreement=("comp_agreement", "max"),  
        subs_agreement=("subs_agreement", "max"),
        cea_agreement = ('cea_agreement', "max"), 
        cea_ceamix_agreement = ('cea_ceamix_agreement', "max"),
        cea_ceas_agreement = ('cea_ceas_agreement', "max"),
        cea_rel_agreement = ('cea_rel_agreement', "max"),
        ce=('ce','max'),
        **{var: (var, "max") for var in features_cluster_columns}
        
    )
)
df_agreements

,conflict_id,year_mo,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,...,ppsex,hrmob,ele,hrni,civso,pubad,juscr,mps,prot,mpspro
0,209,1992-09-01,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,209,1994-06-01,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,209,1995-02-01,1,0,1,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
3,209,1995-06-01,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,209,1996-06-01,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1313,169170,1995-10-01,1,0,1,0,0,0,0,3,...,0,0,1,0,0,0,1,0,1,0
1314,169170,2000-12-01,1,0,0,0,0,0,0,2,...,0,0,0,0,1,0,0,0,0,0
1315,169170,2002-08-01,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1316,169170,2002-10-01,1,0,0,0,0,0,0,2,...,0,0,0,0,1,0,0,0,1,0


In [23]:
df_pax = df_pax.drop_duplicates(subset=['conflict_id', 'year_mo'])
relevant_columns = [i for i in df_pax.columns if not i.startswith('g') and i not in ['agreement', 'comp_agreement', 'subs_agreement', 
                                                                                     'cea_agreement', 'cea_ceas_agreement', 
                                                                                     'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']+features_cluster_columns]
df_pax = df_pax[relevant_columns]
df_agreements = df_agreements.merge(
    df_pax,
    on=['conflict_id', 'year_mo'],
    how='left'    
)
df_agreements.conflict_id.nunique()

115

In [24]:
df_ucdp = pd.read_csv('../../data/input/ucdp/GEDEvent_v25_1.csv', low_memory=False)
#filter the conflicts that only sign agreements
df_ucdp = df_ucdp[df_ucdp['type_of_violence']==1].reset_index(drop=True).copy()
relevant_columns = ['country','conflict_new_id', 'date_start','date_end','best', 'dyad_new_id', 'type_of_violence', 'region', 'conflict_name', 'side_a', 'side_b']
df_ucdp = df_ucdp[relevant_columns].sort_values(['country', 'conflict_new_id', 'date_start'])

#join with isocode and add isocode
isocodes = pd.read_csv('../../data/input/isocodes/isocodes_appended.csv', sep = ';')
df_ucdp = df_ucdp.merge(isocodes[['name', 'alpha_3']], left_on='country', right_on='name', how='left').rename(columns = {'alpha_3':'isocode'})
df_ucdp_conflicts = df_ucdp['conflict_new_id'].unique()

#compare which of the conflicts in df_agreements is not in df_ucdp_conflicts
conflicts_agreements = df_agreements['conflict_id'].unique()
conflicts_not_in_ucdp = set(conflicts_agreements) - set(df_ucdp_conflicts)
print(f"Conflicts in agreements dataset but not in UCDP: {conflicts_not_in_ucdp}")


Conflicts in agreements dataset but not in UCDP: {np.int64(481), np.int64(11745), np.int64(296), np.int64(361), np.int64(235), np.int64(301), np.int64(4750), np.int64(305), np.int64(169170), np.int64(345), np.int64(346)}


In [25]:
len(conflicts_not_in_ucdp)

11

In [26]:
#now print how many conflict intersect 
conflicts_intersect = set(conflicts_agreements) & set(df_ucdp_conflicts)
print(f"Conflicts in both datasets: {len(conflicts_intersect)}")

Conflicts in both datasets: 104


In [16]:
#how many conflicts in df_agreements have year_mo missing
missing_year_mo = df_agreements[df_agreements['year_mo'].isnull()].shape[0]
missing_year_mo

0

In [28]:
df_agreements.conflict_id.unique()

array([   209,    218,    221,    222,    223,    224,    230,    231,
          233,    234,    235,    251,    253,    260,    264,    265,
          267,    269,    271,    274,    283,    287,    288,    289,
          296,    298,    299,    300,    301,    305,    307,    308,
          309,    314,    315,    316,    322,    326,    327,    329,
          330,    331,    332,    333,    335,    336,    337,    341,
          342,    345,    346,    347,    352,    361,    365,    366,
          369,    371,    372,    373,    374,    375,    379,    381,
          382,    384,    385,    386,    387,    388,    389,    390,
          392,    393,    394,    395,    397,    398,    400,    401,
          402,    403,    404,    405,    407,    408,    409,    410,
          411,    412,    413,    416,    417,    419,    420,    421,
          422,    423,    425,    426,    435,    439,    481,   4750,
        11344,  11345,  11346,  11347,  11475,  11745,  13243,  13306,
      

In [33]:
df_pax.loc[df_pax['conflict_id'].isin([298, 373, 379, 223, 274, 322, 347, 365, 425])].groupby('conflict_id').agg(
                                                                                               year_mo_unique = ('year_mo', 'unique'),
                                                                                               ucdp_agreement_id = ('conflict_id', 'unique'))

,year_mo_unique,ucdp_agreement_id
conflict_id,,
223,[2012-04-01 00:00:00],[223]
274,"[1993-09-01 00:00:00, 1996-11-01 00:00:00, 200...",[274]
298,"[1991-05-01 00:00:00, 1992-03-01 00:00:00, 199...",[298]
322,[1997-12-01 00:00:00],[322]
347,"[2015-06-01 00:00:00, 2020-08-01 00:00:00]",[347]
365,"[1995-04-01 00:00:00, 2021-09-01 00:00:00]",[365]
373,"[1994-10-01 00:00:00, 1995-04-01 00:00:00, 199...",[373]
379,"[1994-12-01 00:00:00, 2000-02-01 00:00:00, 200...",[379]
425,[2016-07-01 00:00:00],[425]


From the conflict with conflict termination == 'Agreement'

<code>[[  287   298   372   373   379 11348]]</code>

Only 11348 is not in df_agreements (PAX), and 287, 372 are excluded when considering the filter of conflict being active in the 2 previous quarters (when reducing from 77 to 66). So it remains to investigate **298, 373, 379**

From the conflicts with conflict termination == 'Ceasefire'

<code>[  223   274   292   315   322   347   354   365   425   430 14074]</code>

292, 354, 430, 14074 are not in PAX agreements, and 315 is excluded when considering the filter of conflict being active in the 2 previous quarters (when reducing from 77 to 66). It remains to investigate **223, 274, 322, 347, 365, 425**


So for the following list we found that:

- **298**: (South Africa) the agreements were signed in ['1991-05-01 00:00:00','1992-03-01 00:00:00', '1992-11-01 00:00:00', '1994-02-01 00:00:00']and in UCDP there is violence recorded in 1989-02, 1989-04, and 1989-05.
- **373** (Niger) the agreements were signed in ['1994-10-01 00:00:00', '1995-04-01 00:00:00', '1997-11-01 00:00:00','1998-08-01 00:00:00']but conflict is only recorded from 1994-01 to 1994-09 and then in 1995-03. 
- **379** (Djibouti) the agreements were signed in ['1994-12-01 00:00:00', '2000-02-01 00:00:00', '2001-05-01 00:00:00'] but conflict is only recorded 1991-01 to 1994-03-01 and then 1995-03 to 1999-08.
- **223**: (Myanmar (Burma)) the agreement was signed in 2014-04 and there is no record of violence in UCDP in that year_mo.
- **274**: (India) the agreements were signed in ['1993-09-01 00:00:00', '1996-11-01 00:00:00', '2005-04-01 00:00:00', '2012-01-01 00:00:00', '2013-10-01 00:00:00', '2020-09-01 00:00:00'] but in UCDP there is only violenced recorded in 2020-06 and 2020-08.
- **322** (Bangladesh) the agreements was signed in ['1997-12-01 00:00:00'] and conflict in UCDP was recorded continusly between 1989 to 1997 and then in 2002 and 2020, 2022 but not in 1997-12.
- **347** ('['India']') the agreements were signed in ['2015-06-01 00:00:00', '2020-08-01 00:00:00'] but conflict is not registered in that time, there are in 2015-03 to 2015-04 and then in 2018-01 to 2018-09.
- **365** (India) the agreements were signed in ['1995-04-01 00:00:00','2021-09-01 00:00:00'] but conflict was registered between 1989 continuously from 1989-01 to 2016-12 but in those particular months there is no record of violence in UCDP.
- **425** (Nigeria) the agreement signed is in 2016, but conflict is only recorded between 2004 and then in 2009.

And then make sure that the conflicts that are not in PAX, why are included in UCDP peace agreements? Are informal peace agreements and that's why PAX doesnt track them. This list is:

<code>[292,354,430,14074,11348]</code>


- El conflict_id 11348 (Sudan - South Sudan) in UCDP has 4 agreements, the first recorded in 20121-09, but in PAX the same agreement is associated to other conflict_id that is 309.

In [38]:
ucdp_agreements = pd.read_excel('../../data/input/ucdp/ucdp-peace-agreements-221.xlsx')
ucdp_agreements.loc[ucdp_agreements['conflict_id'].isin(['292', '354', '430', '14074', '11348'])]

,paid,region,gwno,conflict_id,conflict_name,dyad_id,dyad_name,actor_id,actor_name,incompatibility,...,linktofulltextagreement,inclusive,no_dyad,pa_type,out_iss,procID,frame,version,dateintervalstart_meta,dateintervalend_meta
244,1036,4,436,430,Niger: Government,895,Government of Niger - FLAA,"75, 524","Government of Niger, FLAA",2,...,NaN,2,1,2,4,56,1,22.1,1975-01-01,2021-12-31
245,1563,4,436,430,Niger: Government,896,Government of Niger - UFRA,"3989, 75, 526","FARS, Government of Niger, UFRA",2,...,http://ucdpged.uu.se/peaceagreements/fulltext/...,1,1,2,1,128,1,22.1,1975-01-01,2021-12-31
298,1444,4,"625, 626",11348,South Sudan - Sudan,11989,Government of South Sudan - Government of Sudan,"7071, 113, 112","IGAD, Government of South Sudan, Government of...",1,...,http://ucdpged.uu.se/peaceagreements/fulltext/...,1,1,2,1,130,1,22.1,1975-01-01,2021-12-31
299,1456,4,"625, 626",11348,South Sudan - Sudan,11989,Government of South Sudan - Government of Sudan,"4231, 113, 112","African Union, Government of South Sudan, Gove...",1,...,http://ucdpged.uu.se/peaceagreements/fulltext/...,1,1,2,1,130,1,22.1,1975-01-01,2021-12-31
300,1457,4,"625, 626",11348,South Sudan - Sudan,11989,Government of South Sudan - Government of Sudan,"4231, 113, 112","African Union, Government of South Sudan, Gove...",1,...,http://ucdpged.uu.se/peaceagreements/fulltext/...,1,1,2,1,130,1,22.1,1975-01-01,2021-12-31
301,1531,4,"625, 626",11348,South Sudan - Sudan,11989,Government of South Sudan - Government of Sudan,"4231, 113, 112","African Union, Government of South Sudan, Gove...",1,...,http://ucdpged.uu.se/peaceagreements/fulltext/...,1,1,2,1,130,3,22.1,1975-01-01,2021-12-31


The conflict <code>14074</code> is not even in UCDP Peace Agreements, so the reason why UCDP is coding it as conflict termination == agreement can be an error.


- Niger (430): UCDP tiene dos acuerdos formales firmados — el Paris Accord (1993) y el Protocole (1997). Están bien documentados, tienen signatarios nombrados, son escritos. PA-X debería tenerlos pero no los tiene. La razón probable es cobertura temporal y geográfica incompleta de PA-X: PA-X es más fuerte en acuerdos de alto perfil internacional y en ciertos conflictos con documentación en inglés. Acuerdos firmados en francés en conflictos africanos de mediana intensidad de los 90 tienen gaps.
- South Sudan - Sudan (11348): UCDP tiene cuatro acuerdos de Addis Abeba (2012) firmados, con signatarios claros y texto disponible. Estos son formalmente acuerdos entre dos estados soberanos (interstate), no un conflicto intraestatal. PA-X cubre principalmente conflictos intraestatales — acuerdos entre un gobierno y grupos armados no estatales. Los acuerdos interstate entre dos gobiernos quedan fuera del scope de PA-X o en sus márgenes.
- 292 y 354: No aparecen ni en UCDP Peace Agreements. Esto significa que UCDP Termination los clasifica como "agreement" o "ceasefire" pero el equipo de UCDP Peace Agreements nunca los codificó como acuerdos formales. Son los casos más interesantes — terminación de facto sin acuerdo formal: el conflicto se apagó, hubo negociaciones o un alto al fuego tácito que UCDP registró como terminación por acuerdo, pero nunca hubo un documento firmado que UCDP PA o PA-X pudieran codificar.

In [43]:
df_pax.agtp.value_counts(normalize=True)*100

agtp
Intra         70.789074
InterIntra    13.201821
IntraLocal    11.532625
Inter          4.476480
Name: proportion, dtype: float64